# 04 Export FIF and run Zuna augmentation

Export preprocessed FIF files, run Zuna augmentation, and build the augmented epoch index.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists():
    if (ROOT.parent / "scripts").exists():
        ROOT = ROOT.parent
    elif (ROOT.parent.parent / "scripts").exists():
        ROOT = ROOT.parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MANIFEST = ROOT / "data/manifests/unified_manifest.csv"
EPOCH_INDEX = ROOT / "data/manifests/epoch_index_yoto_tones.csv"
AUG_INDEX = ROOT / "data/manifests/epoch_index_yoto_tones_zuna.csv"

FIF_DIR = ROOT / "data/processed/fif_for_zuna"
AUG_OUT_DIR = ROOT / "data/processed/epochs_aug"
ZUNA_WORK_DIR = ROOT / "runs/zuna_yoto"

DATASETS = ["ds005815"]
TASK_CONTAINS = "task-task"
SKIP_ASR = True
SKIP_ICA = True

SUBJECT_ID = ""  # Optional filter, e.g. "sub-01"
GPU_DEVICE = "0"
TOKENS_PER_BATCH = 2048
DIFFUSION_STEPS = 8

In [ ]:
from scripts.build_zuna_augmented_index_yoto import build_zuna_epoch_index
from scripts.export_yoto_to_fif import export_yoto_fif
from scripts.run_zuna_augmentation import run_zuna_augmentation

fif_paths = export_yoto_fif(
    manifest=MANIFEST,
    datasets=DATASETS,
    task_contains=TASK_CONTAINS,
    out_dir=FIF_DIR,
    skip_asr=SKIP_ASR,
    skip_ica=SKIP_ICA,
)
print(f"Exported FIF files: {len(fif_paths)}")

zuna_info = run_zuna_augmentation(
    input_fif_dir=FIF_DIR,
    work_dir=ZUNA_WORK_DIR,
    subject_id=SUBJECT_ID,
    gpu_device=GPU_DEVICE,
    tokens_per_batch=TOKENS_PER_BATCH,
    diffusion_steps=DIFFUSION_STEPS,
)

rows = build_zuna_epoch_index(
    zuna_fif_dir=zuna_info["fif_output_dir"],
    epoch_index=EPOCH_INDEX,
    out_dir=AUG_OUT_DIR,
    out_index=AUG_INDEX,
)
print(f"Augmented epoch rows: {len(rows)}")